# AGRO AI — Production Machine Learning Training Notebook

**Project:** AGRO AI — Smart Agriculture. Healthy Lives. Better Future.
**Objective:** End-to-end reproducible training pipeline for:
1. Agricultural Relevance Image Classifier (Agriculture vs Non-Agriculture)
2. Multi-Class Crop Foliar Disease Classifier (MobileNetV2 CNN)
3. Visual Soil Category Classifier

---

## Cell 1: Environment Setup & Hardware Acceleration Check

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn pillow numpy pandas

import os
import sys
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print("TensorFlow Version:", tf.__version__)
gpu_devices = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpu_devices if gpu_devices else "No GPU detected (Running on CPU)")

## Cell 2: Dataset Acquisition & Directory Setup

In [ ]:
# Create expected directory structures
BASE_DIR = Path("./agro_ai_datasets")
CROP_DIR = BASE_DIR / "crop_disease"
RELEVANCE_DIR = BASE_DIR / "agricultural_relevance"
SOIL_DIR = BASE_DIR / "soil"

for p in [CROP_DIR / "train", CROP_DIR / "validation", CROP_DIR / "test",
          RELEVANCE_DIR / "agriculture", RELEVANCE_DIR / "non_agriculture",
          SOIL_DIR / "train", SOIL_DIR / "validation", SOIL_DIR / "test"]:
    p.mkdir(parents=True, exist_ok=True)

print("Directory structure prepared at:", BASE_DIR)

## Cell 3: Dataset Inspection & Summary

In [ ]:
def inspect_dataset_tree(target_dir):
    summary = {}
    for root, dirs, files in os.walk(target_dir):
        img_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        if img_files:
            class_name = os.path.basename(root)
            summary[class_name] = len(img_files)
    return summary

print("Crop Training Categories:", inspect_dataset_tree(CROP_DIR / 'train'))

## Cell 4: Class Distribution Visualization

In [ ]:
def plot_class_distribution(dist_dict, title="Dataset Class Balance"):
    if not dist_dict:
        print("No classes found yet in directory.")
        return
    plt.figure(figsize=(12, 5))
    sns.barplot(x=list(dist_dict.keys()), y=list(dist_dict.values()), palette="viridis")
    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.ylabel("Image Count")
    plt.tight_layout()
    plt.show()

## Cell 5: Image Integrity Verification

In [ ]:
def check_image_integrity(folder_path):
    corrupted_files = []
    valid_count = 0
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                f_path = Path(root) / f
                try:
                    with Image.open(f_path) as img:
                        img.verify()
                    valid_count += 1
                except Exception:
                    corrupted_files.append(f_path)
    print(f"Integrity check for {folder_path}: {valid_count} valid images, {len(corrupted_files)} corrupted images.")
    return corrupted_files

## Cell 6: Corrupted Image Handling & Quarantine

In [ ]:
def quarantine_corrupted(corrupted_list):
    quarantine_dir = Path("./quarantine")
    quarantine_dir.mkdir(exist_ok=True)
    for f in corrupted_list:
        try:
            f.rename(quarantine_dir / f.name)
            print(f"Quarantined: {f.name}")
        except Exception as e:
            print(f"Error quarantining {f}: {e}")

## Cell 7: Duplicate Image Detection via MD5 Hashing

In [ ]:
def detect_duplicate_images(folder_path):
    hashes = {}
    duplicates = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                f_path = Path(root) / f
                with open(f_path, "rb") as fp:
                    h = hashlib.md5(fp.read()).hexdigest()
                if h in hashes:
                    duplicates.append((f_path, hashes[h]))
                else:
                    hashes[h] = f_path
    print(f"Discovered {len(duplicates)} exact duplicate image pairs in {folder_path}.")
    return duplicates

## Cell 8: Image Preprocessing Pipeline

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def preprocess_input_image(image_path):
    img = Image.open(image_path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.Resampling.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    return np.expand_dims(arr, axis=0)

## Cell 9: Train / Validation / Test Splitting

In [ ]:
# Example utility to split a raw directory into 70% Train, 15% Val, 15% Test
def create_stratified_split(raw_dir, output_base, train_pct=0.70, val_pct=0.15):
    import shutil
    for class_folder in Path(raw_dir).iterdir():
        if class_folder.is_dir():
            files = list(class_folder.glob("*.jpg")) + list(class_folder.glob("*.png"))
            np.random.shuffle(files)
            n_train = int(len(files) * train_pct)
            n_val = int(len(files) * val_pct)
            
            train_files = files[:n_train]
            val_files = files[n_train:n_train+n_val]
            test_files = files[n_train+n_val:]
            
            for subset, f_list in [("train", train_files), ("validation", val_files), ("test", test_files)]:
                dest = output_base / subset / class_folder.name
                dest.mkdir(parents=True, exist_ok=True)
                for f in f_list:
                    shutil.copy(f, dest / f.name)

## Cell 10: Data Augmentation Configuration

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

## Cell 11: Transfer Learning Architecture (MobileNetV2)

In [ ]:
def build_agro_mobilenet_model(num_classes):
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
    )
    
    # Freeze initial feature extraction layers
    for layer in base_model.layers[:-25]:
        layer.trainable = False
        
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=outputs)
    return model

## Cell 12: Model Compilation & Hyperparameters

In [ ]:
NUM_CLASSES = 17  # Standard PlantVillage subset
model = build_agro_mobilenet_model(NUM_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)
model.summary()

## Cell 13: Model Training with Callbacks

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint('best_crop_disease_model.keras', monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

# Note: Call model.fit with your generators
# history = model.fit(train_generator, validation_data=val_generator, epochs=20, callbacks=callbacks)

## Cell 14: Validation Curves & Training History

In [ ]:
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    plt.show()

## Cell 15: Evaluation on Unseen Test Dataset

In [ ]:
# test_loss, test_acc, test_top3 = model.evaluate(test_generator)
# print(f"Test Accuracy: {test_acc*100:.2f}%")
# print(f"Test Top-3 Accuracy: {test_top3*100:.2f}%")

## Cell 16: Classification Report (Precision, Recall, F1-Score)

In [ ]:
def generate_evaluation_report(model, test_gen):
    y_true = test_gen.classes
    preds = model.predict(test_gen)
    y_pred = np.argmax(preds, axis=1)
    labels = list(test_gen.class_indices.keys())
    
    print("--- CLASSIFICATION REPORT ---")
    print(classification_report(y_true, y_pred, target_names=labels))
    return y_true, y_pred, labels

## Cell 17: Confusion Matrix Heatmap

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
    plt.title('Test Dataset Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Cell 18: Model Saving (.keras and .h5)

In [ ]:
model.save('crop_disease.keras')
model.save('crop_disease.h5')
print("✓ Model successfully saved in both .keras and .h5 formats.")

## Cell 19: Export Metadata JSON

In [ ]:
metadata = {
    "model_name": "Crop Pathogen & Foliage Classifier",
    "version": "1.0.0",
    "architecture": "MobileNetV2 Fine-Tuned CNN",
    "input_shape": [224, 224, 3],
    "dataset_source": "PlantVillage & PlantDoc Annotated Datasets",
    "training_date": str(pd.Timestamp.now().date()),
    "target_classes": NUM_CLASSES
}

with open('crop_disease_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✓ Metadata exported to crop_disease_metadata.json")

## Cell 20: Inference Testing on Newly Uploaded Images

In [ ]:
def test_single_inference(model, image_path, class_names):
    processed = preprocess_input_image(image_path)
    pred = model.predict(processed, verbose=0)[0]
    top_idx = np.argmax(pred)
    confidence = float(pred[top_idx])
    
    predicted_label = class_names[top_idx] if top_idx < len(class_names) else "Unknown"
    print(f"Predicted Class: {predicted_label} | Confidence: {confidence*100:.2f}%")
    return predicted_label, confidence